# QLoRA Fine-Tuning

Fine-tunes **Llama-3.2-3B-Instruct** with QLoRA (4-bit base model + LoRA adapters) on the
Salesforce Apex/SOQL/LWC instruction dataset (`data/processed/train.jsonl` /
`val.jsonl`, 275 / 31 examples).

**Before running:**
1. Llama-3.2 is a gated model on Hugging Face. Request access at
   https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and create a token with
   `Read` access, then add it as a Colab secret named `HF_TOKEN` (key icon in the left
   sidebar).
2. Use a GPU runtime (Runtime -> Change runtime type -> T4 GPU, or better).
3. Upload `data/processed/train.jsonl` and `data/processed/val.jsonl` from the repo into
   this Colab session's working directory (left sidebar -> Files -> upload) before
   running the data-loading cell.

**LoRA config, learning rate, epochs, and batch size are all set as plain variables in the
"Config" cell below** — edit those to experiment without touching the rest of the
notebook.

In [ ]:
# Pin versions to avoid API breakage from `pip install -U` picking up a newer trl/peft
# that changes the SFTTrainer/SFTConfig signature. bitsandbytes is intentionally left
# unpinned (>=0.45.0 floor) since it needs to match whatever CUDA version Colab's
# current image ships -- an older pinned build can be missing a prebuilt CUDA binary
# and fall back to a broken CPU-only path.
!pip install -q transformers==4.44.2 accelerate==0.33.0 "bitsandbytes>=0.45.0" \
    peft==0.12.0 trl==0.9.6 datasets==2.20.0 sentencepiece

In [ ]:
import time
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./qlora-checkpoints"
ADAPTER_DIR = "./lora-adapter"

# --- Config (Story C1: configurable LoRA / training hyperparameters) ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024

## Load the dataset

Upload `train.jsonl` and `val.jsonl` (from `data/processed/`) into this Colab session's
working directory first.

In [ ]:
dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "val.jsonl"},
)
print(dataset)
print(dataset["train"][0])

In [ ]:
def formatting_func(examples):
    """Format a *batch* of {instruction, input, output} examples using the Llama-3.2
    chat template. SFTTrainer calls this with `batched=True`, so every field is a list
    of values rather than a single value -- this must return a list of formatted
    strings, one per example in the batch."""
    output_texts = []
    for i in range(len(examples["instruction"])):
        user_content = examples["instruction"][i]
        if examples.get("input") and examples["input"][i]:
            user_content += "\n\n" + examples["input"][i]

        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": examples["output"][i]},
        ]
        output_texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return output_texts

## Load the base model in 4-bit

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Configure and run training

`report_to` is set to `"none"` by default. To log to Weights & Biases instead, set
`report_to="wandb"` and add a `WANDB_API_KEY` Colab secret — `wandb.init()` will prompt for
it automatically on first use.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

In [ ]:
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0

print(f"Training took {train_time / 60:.1f} min")
print(train_result)

## Inspect the loss curve

Pulls train/eval loss from the trainer's log history and plots it — paste the resulting
numbers/plot into the DEVLOG for Story C2.

In [ ]:
import matplotlib.pyplot as plt

train_loss = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]

if train_loss:
    steps, losses = zip(*train_loss)
    plt.plot(steps, losses, label="train_loss")
if eval_loss:
    steps, losses = zip(*eval_loss)
    plt.plot(steps, losses, marker="o", label="eval_loss")

plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.title("QLoRA fine-tuning loss")
plt.show()

print("train_loss:", train_loss)
print("eval_loss:", eval_loss)

## Save the LoRA adapter

Saves the adapter (small — just the LoRA weights, not the base model) and zips it for
download. Copy/unzip it into `models/lora-adapter/` in the repo (gitignored — large
binary files stay out of git).

In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

!zip -r lora-adapter.zip {ADAPTER_DIR}
print("Saved and zipped:", ADAPTER_DIR)

In [ ]:
from google.colab import files
files.download("lora-adapter.zip")

## Notes

After running, fill in below for the DEVLOG (Story C2):

- **Load time:** _TBD_
- **GPU memory allocated:** _TBD_
- **Training time:** _TBD_
- **Final train loss / eval loss:** _TBD_
- **Any compatibility issues encountered:** _TBD_
- **LoRA config used (r / alpha / dropout / target modules):** _TBD_